# BERT Encoder-Based Sentiment Analysis (IMDB)

Fine-tune `bert-base-uncased` to classify movie reviews as **positive** or **negative**.

**Pipeline:** text → tokenizer → BERT encoder (12 layers) → `[CLS]` vector → linear layer → positive / negative

> Tip: run on a GPU (Colab / Kaggle: Runtime → Change runtime type → GPU).

## 1. Setup

In [ ]:
!pip install -q transformers datasets evaluate scikit-learn accelerate seaborn

In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import evaluate
from datasets import load_dataset
from sklearn.metrics import confusion_matrix, classification_report
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, Trainer, TrainingArguments, pipeline,
)

MODEL_NAME = "bert-base-uncased"
MAX_LEN = 256
SEED = 42
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

## 2. Load data & quick EDA

In [ ]:
dataset = load_dataset("imdb")
print(dataset)

In [ ]:
train_df = dataset["train"].to_pandas()
train_df["n_words"] = train_df["text"].str.split().str.len()

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
train_df["label"].map({0: "Negative", 1: "Positive"}).value_counts().plot.bar(ax=ax[0], color=["#4C78A8", "#F58518"])
ax[0].set_title("Class balance (train)")
ax[1].hist(train_df["n_words"], bins=50, color="#54A24B")
ax[1].axvline(MAX_LEN, color="red", ls="--", label=f"MAX_LEN = {MAX_LEN} tokens")
ax[1].set_title("Review length (words)"); ax[1].legend()
plt.tight_layout(); plt.show()
print(train_df["n_words"].describe())

## 3. Tokenization

The tokenizer splits text into WordPiece tokens, adds `[CLS]` at the start and `[SEP]` at the end, and maps tokens to IDs.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

sample = "The movie was not good."
enc = tokenizer(sample)
print(tokenizer.convert_ids_to_tokens(enc["input_ids"]))
print(enc["input_ids"])

In [ ]:
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

tokenized = dataset.map(tokenize, batched=True)
collator = DataCollatorWithPadding(tokenizer=tokenizer)  # pads dynamically per batch

# Hold out part of train for validation so the test set stays untouched
split = tokenized["train"].train_test_split(test_size=0.1, seed=SEED)
train_ds, val_ds, test_ds = split["train"], split["test"], tokenized["test"]
print(len(train_ds), len(val_ds), len(test_ds))

## 4. Model

`AutoModelForSequenceClassification` = BERT encoder + dropout + a linear layer on the `[CLS]` representation, giving 2 logits.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2,
    id2label={0: "NEGATIVE", 1: "POSITIVE"},
    label2id={"NEGATIVE": 0, "POSITIVE": 1},
)
print(f"Parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

## 5. Training

In [ ]:
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1.compute(predictions=preds, references=labels)["f1"],
    }

args = TrainingArguments(
    output_dir="bert-sentiment",
    learning_rate=2e-5,              # small LR keeps pre-trained knowledge intact
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(),
    seed=SEED,
    report_to="none",
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds,
    processing_class=tokenizer, data_collator=collator,
    compute_metrics=compute_metrics,
)
trainer.train()

In [ ]:
logs = pd.DataFrame(trainer.state.log_history)
loss = logs.dropna(subset=["loss"])
plt.plot(loss["step"], loss["loss"]); plt.xlabel("step"); plt.ylabel("train loss")
plt.title("Training loss"); plt.show()

## 6. Evaluation on the test set

In [ ]:
out = trainer.predict(test_ds)
y_pred = np.argmax(out.predictions, axis=-1)
y_true = out.label_ids

print(out.metrics)
print(classification_report(y_true, y_pred, target_names=["Negative", "Positive"], digits=4))

cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Neg", "Pos"], yticklabels=["Neg", "Pos"])
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion matrix"); plt.show()

## 7. Save model & run inference

In [ ]:
trainer.save_model("bert-sentiment/final")
tokenizer.save_pretrained("bert-sentiment/final")

clf = pipeline("text-classification", model="bert-sentiment/final",
               device=0 if torch.cuda.is_available() else -1)

texts = [
    "A brilliant, moving film. Loved every minute.",
    "Dull plot and terrible acting. Total waste of time.",
    "The movie was not good.",
    "Not bad at all, actually quite enjoyable!",
]
for t, r in zip(texts, clf(texts)):
    print(f"{r['label']:9} ({r['score']:.2%})  {t}")

## 8. Conclusion & next steps

- Fine-tuned BERT typically reaches ~92–93% accuracy on IMDB after 2 epochs.
- Because BERT is bidirectional, it handles negation ("not good") better than bag-of-words models.

**Ideas to improve:** raise `MAX_LEN` to 512, try `roberta-base` or `distilbert-base-uncased` (faster), tune learning rate / epochs, or inspect misclassified reviews for error analysis.